# Constraint-aware Mask-Filling trial 4

This notebook expands rule-based variation while preserving the `mask-trial2` / `mask-trial3` CSV format. It classifies DATE values into absolute dates, month/day values, gestational ages, week ranges, and durations. Other protected fields continue to use the deterministic filling rules in `src/mask_filling_trial4.py`.

In [ ]:
# Colab only: uncomment when the packages are not installed.
# %pip install -q pandas torch transformers


In [ ]:
from __future__ import annotations

import calendar
import json
import logging
import os
import re
import sys
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
LOGGER = logging.getLogger("mask_filling_trial4_notebook")

REPO_DIR = Path(os.getenv("HECTA_REPO_DIR", Path.cwd())).resolve()
SRC_DIR = REPO_DIR / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"src directory was not found: {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import mask_filling_trial3 as trial3
import mask_filling_trial4 as trial4


## Runtime policy
Paths can be supplied through environment variables so that local and Colab runs use the same notebook.

In [ ]:
INPUT_CSV = Path(os.getenv("MASK_FILLING_INPUT_CSV", REPO_DIR / "clinical_case_mask_trial.csv"))
OUTPUT_DIR = Path(os.getenv("MASK_FILLING_OUTPUT_DIR", REPO_DIR / "mask_filling_outputs"))
MODEL_NAME = os.getenv("MASK_FILLING_MODEL_NAME", "emilyalsentzer/Bio_ClinicalBERT")
DEVICE = os.getenv("MASK_FILLING_DEVICE", "auto")

TOP_K = 12
MAX_ROWS: int | None = None
MAX_MASKS_PER_ROW: int | None = None
AUGMENTATION_FACTOR = 5
CHANGE_SEX = True
DATE_SHIFT_MIN_DAYS = 30
DATE_SHIFT_MAX_DAYS = 365
GESTATIONAL_WEEK_DELTA = 2
DURATION_DELTA = 2

print(f"Input : {INPUT_CSV}")
print(f"Output: {OUTPUT_DIR}")

## Extended DATE rules

Absolute dates use one row-level offset so chronology is preserved. Month/day values use the same offset with a neutral leap year internally. Week ranges, gestational ages, and durations receive small bounded changes rather than calendar-date shifts.

In [ ]:
ORIGINAL_REPLACE_DATE = trial3.replace_date
MONTH_DAY_PATTERN = re.compile(r"^(?P<month>[A-Za-z]{3,9})\s+(?P<day>\d{1,2})(?P<suffix>st|nd|rd|th)?$", re.IGNORECASE)
WEEK_RANGE_PATTERN = re.compile(r"^(?P<start>\d{1,2})\s*[-–]\s*(?P<end>\d{1,2})\s+weeks?$", re.IGNORECASE)
GESTATIONAL_PATTERN = re.compile(r"^(?P<prefix>(?:gestational\s+)?age\s+)(?P<weeks>\d{1,2})(?P<suffix>\s+weeks?)$", re.IGNORECASE)
DURATION_PATTERN = re.compile(r"^(?P<number>\d{1,3})(?P<space>\s*)(?P<unit>days?|weeks?|months?|years?)$", re.IGNORECASE)

def classify_date_value(value: str) -> str:
    """Classify a DATE mask into a machine-actionable replacement policy."""
    if trial3.parse_date_value(value)[0] is not None:
        return "absolute_date"
    if MONTH_DAY_PATTERN.fullmatch(value.strip()):
        return "month_day"
    if WEEK_RANGE_PATTERN.fullmatch(value.strip()):
        return "week_range"
    if GESTATIONAL_PATTERN.fullmatch(value.strip()):
        return "gestational_age"
    if DURATION_PATTERN.fullmatch(value.strip()):
        return "duration"
    return "unparsed"

def bounded_delta(seed_days: int, limit: int) -> int:
    """Create a reproducible non-zero small delta from the row date offset."""
    magnitude = abs(seed_days) % max(1, limit) + 1
    return magnitude if seed_days >= 0 else -magnitude

def shift_month_day(value: str, date_shift_days: int) -> str:
    """Shift a month/day value while retaining its year-free display style."""
    match = MONTH_DAY_PATTERN.fullmatch(value.strip())
    if not match:
        return value
    try:
        parsed = datetime.strptime(f"{match.group('month')} {match.group('day')} 2000", "%B %d %Y")
    except ValueError:
        try:
            parsed = datetime.strptime(f"{match.group('month')} {match.group('day')} 2000", "%b %d %Y")
        except ValueError:
            LOGGER.warning("Could not parse month/day DATE value: %s", value)
            return value
    shifted = parsed + timedelta(days=date_shift_days)
    month = calendar.month_name[shifted.month]
    return f"{month} {shifted.day}"

def shift_week_range(value: str, date_shift_days: int) -> str:
    """Move a week range by a small amount while preserving its width."""
    match = WEEK_RANGE_PATTERN.fullmatch(value.strip())
    if not match:
        return value
    start, end = int(match.group('start')), int(match.group('end'))
    delta = bounded_delta(date_shift_days, GESTATIONAL_WEEK_DELTA)
    new_start = max(1, start + delta)
    new_end = new_start + (end - start)
    return f"{new_start}-{new_end} weeks"

def shift_gestational_age(value: str, date_shift_days: int) -> str:
    """Adjust gestational age within a small configured range."""
    match = GESTATIONAL_PATTERN.fullmatch(value.strip())
    if not match:
        return value
    weeks = int(match.group('weeks'))
    delta = bounded_delta(date_shift_days, GESTATIONAL_WEEK_DELTA)
    return f"{match.group('prefix')}{max(1, min(42, weeks + delta))}{match.group('suffix')}"

def shift_duration(value: str, date_shift_days: int) -> str:
    """Adjust a relative duration without changing its unit."""
    match = DURATION_PATTERN.fullmatch(value.strip())
    if not match:
        return value
    number = int(match.group('number'))
    delta = bounded_delta(date_shift_days, DURATION_DELTA)
    return f"{max(1, number + delta)}{match.group('space')}{match.group('unit')}"

def replace_date_extended(original_value: str, date_shift_days: int) -> str:
    """Apply the rule selected for a DATE-like value."""
    date_kind = classify_date_value(original_value)
    handlers = {
        "absolute_date": ORIGINAL_REPLACE_DATE,
        "month_day": shift_month_day,
        "week_range": shift_week_range,
        "gestational_age": shift_gestational_age,
        "duration": shift_duration,
    }
    handler = handlers.get(date_kind)
    if handler is None:
        LOGGER.warning("Unsupported DATE value was restored: %s", original_value)
        return original_value
    return handler(original_value, date_shift_days)


## Enrich machine-readable metadata

The original `mask_metadata` remains compatible. This step adds explicit policies used for auditing and stores them in `enriched_mask_metadata`.

In [ ]:
def enrich_metadata(raw_metadata: str) -> str:
    """Add explicit replacement settings without discarding source metadata."""
    enriched: list[dict[str, Any]] = []
    for source_item in trial3.parse_metadata(raw_metadata):
        item = dict(source_item)
        mask_type = str(item.get("mask_type", "LOW_RISK_WORDING"))
        original = str(item.get("original_value", ""))
        if mask_type == "DATE":
            date_kind = classify_date_value(original)
            item.update({
                "date_kind": date_kind,
                "replacement_policy": f"shift_{date_kind}" if date_kind != "unparsed" else "preserve_unparsed",
                "preserve_chronology": date_kind in {"absolute_date", "month_day"},
            })
        elif mask_type == "SEX":
            item.update({"replacement_policy": "swap_consistently" if CHANGE_SEX else "preserve", "consistency_scope": "row"})
        elif mask_type == "AGE":
            item["replacement_policy"] = "same_age_category"
        elif mask_type in {"NAME", "LOCATION", "ORGANIZATION", "ID"}:
            item["replacement_policy"] = "fictitious_value"
        else:
            item["replacement_policy"] = "clinicalbert_filtered"
        enriched.append(item)
    return json.dumps(enriched, ensure_ascii=False)

# trial4 delegates DATE replacement to trial3; replace that hook for this run.
trial3.replace_date = replace_date_extended


## Run filling

In [ ]:
try:
    input_frame = trial3.load_mask_trial2_csv(INPUT_CSV)
    input_frame["enriched_mask_metadata"] = input_frame["mask_metadata"].map(enrich_metadata)
    # Use enriched metadata for filling while preserving the original column in the output.
    input_frame["source_mask_metadata"] = input_frame["mask_metadata"]
    input_frame["mask_metadata"] = input_frame["enriched_mask_metadata"]
except Exception:
    LOGGER.exception("Failed to prepare the Mask-Filling input")
    raise

config = trial3.FillingConfig(
    model_name=MODEL_NAME,
    device=DEVICE,
    top_k=TOP_K,
    max_masks_per_row=MAX_MASKS_PER_ROW,
    augmentation_factor=AUGMENTATION_FACTOR,
    change_sex=CHANGE_SEX,
    date_shift_min_days=DATE_SHIFT_MIN_DAYS,
    date_shift_max_days=DATE_SHIFT_MAX_DAYS,
)

filler = trial3.ClinicalBertFiller(config.model_name, config.device)
filled_frame = trial4.run_mask_filling(input_frame, MAX_ROWS, config, filler)
print(f"Generated rows: {len(filled_frame)}")
filled_frame[["source_row_id", "variant_id", "fill_status", "validation_issues"]].head()

## Save and inspect results

In [ ]:
try:
    output_path = trial4.save_output(filled_frame, OUTPUT_DIR)
except Exception:
    LOGGER.exception("Failed to save Mask-Filling results")
    raise

print(f"CSV saved: {output_path}")
display_columns = ["source_row_id", "variant_id", "original_text", "filled_text", "fill_status", "validation_issues"]
filled_frame[[column for column in display_columns if column in filled_frame.columns]].head(10)

## Checks

Confirm that: (1) `fill_status` is `filled`, (2) `validation_issues` is `[]`, (3) all absolute dates in one case use the same offset, (4) week ranges retain their width, and (5) SEX terms are consistently preserved or swapped across a row. Unsupported DATE expressions are restored and logged rather than guessed.